In [ ]:
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import joblib
# ---------------- CONFIG ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_CLASSES = 7  # Modify according to your dataset

In [ ]:
# ---------------- LOAD TRAINED MODEL ----------------
def build_model(num_classes):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )
    return model



In [ ]:
model = build_model(NUM_CLASSES)
model.load_state_dict(torch.load("cnn_best.pth", map_location=DEVICE))
model.eval().to(DEVICE)

# ---------------- FEATURE EXTRACTOR ----------------
feature_extractor = nn.Sequential(*list(model.children())[:-1])
feature_extractor.eval().to(DEVICE)

# ---------------- FEATURE EXTRACTION ----------------
def extract_features(dataloader, extractor):
    features, labels = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(dataloader, desc="Extracting features"):
            imgs = imgs.to(DEVICE)
            feats = extractor(imgs).squeeze()  # (B, 512, 1, 1) → (B, 512)
            if len(feats.shape) > 2:
                feats = feats.view(feats.size(0), -1)
            features.append(feats.cpu().numpy())
            labels.extend(lbls.numpy())
    return np.vstack(features), np.array(labels)



In [ ]:
# ---------------- TRANSFORMS ----------------
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ---------------- LOAD DATASET ----------------
DATA_DIR = "/kaggle/input/fer2013"
train_data = datasets.ImageFolder(f"{DATA_DIR}/train", transform=eval_transform)
test_data = datasets.ImageFolder(f"{DATA_DIR}/test", transform=eval_transform)

# ---------------- DATALOADERS ----------------
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# ---------------- EXTRACT FEATURES ----------------
X_train, y_train = extract_features(train_loader, feature_extractor)
X_test, y_test = extract_features(test_loader, feature_extractor)
# ---------------- SCALING ----------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# ---------------- TRAIN LOGISTIC REGRESSION ----------------
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)

# ---------------- PREDICT ON TEST SET ----------------
y_pred_test = logreg.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred_test)
print(f"✅ Test Accuracy: {accuracy:.4f}")

# ---------------- SAVE MODEL AND SCALER ----------------
joblib.dump(logreg, "logistic_regression_model.pkl")
joblib.dump(scaler, "feature_scaler.pkl")

print("✅ Saved Logistic Regression Model and Scaler")


In [ ]:
# ---------------- LOAD AND PREDICT AGAIN (Example of loading) ----------------
loaded_logreg = joblib.load("logistic_regression_model.pkl")
loaded_scaler = joblib.load("feature_scaler.pkl")

# Example on how to use the loaded model for new predictions
new_features, _ = extract_features(test_loader, feature_extractor)
new_features_scaled = loaded_scaler.transform(new_features)
new_predictions = loaded_logreg.predict(new_features_scaled)

print(f"✅ Example Prediction (first 10): {new_predictions[:10]}")
